**Self-Attention:**
- Starting with a basic form of self-attention

In [1]:
import torch

# assuming we have this sentence below tokenized..

sentence = torch.tensor([
    0, # can
    7, # you
    1, # help
    2, # me
    5, # to
    6, # translate
    4, # this 
    3 # sentence
])

sentence

tensor([0, 7, 1, 2, 5, 6, 4, 3])

Lets also assume we already encoded this sentence into a real-number vector representation via an embedding layer. Here our embedding size is 16, and we assume that the dictionary size is 10. The following code will then produce the word embeddings of our 8 words:

In [3]:
torch.manual_seed(123)

embed = torch.nn.Embedding(10, 16)

embedded_sentence = embed(sentence).detach()
embedded_sentence.shape

torch.Size([8, 16])

Now, we can assume that the above is the input (batch size of 1) - and can compute the dot product scores between the ith and jth word embeddings. We can do this for all wij values as follows:

In [4]:
embedded_sentence

tensor([[ 3.3737e-01, -1.7778e-01, -3.0353e-01, -5.8801e-01,  3.4861e-01,
          6.6034e-01, -2.1964e-01, -3.7917e-01,  7.6711e-01, -1.1925e+00,
          6.9835e-01, -1.4097e+00,  1.7938e-01,  1.8951e+00,  4.9545e-01,
          2.6920e-01],
        [-9.4053e-01, -4.6806e-01,  1.0322e+00, -2.8300e-01,  4.9275e-01,
         -1.4078e-02, -2.7466e-01, -7.6409e-01,  1.3966e+00, -9.9491e-01,
         -1.5822e-03,  1.2471e+00, -7.7105e-02,  1.2774e+00, -1.4596e+00,
         -2.1595e+00],
        [-7.7020e-02, -1.0205e+00, -1.6896e-01,  9.1776e-01,  1.5810e+00,
          1.3010e+00,  1.2753e+00, -2.0095e-01,  4.9647e-01, -1.5723e+00,
          9.6657e-01, -1.1481e+00, -1.1589e+00,  3.2547e-01, -6.3151e-01,
         -2.8400e+00],
        [-1.3250e+00,  1.7843e-01, -2.1338e+00,  1.0524e+00, -3.8848e-01,
         -9.3435e-01, -4.9914e-01, -1.0867e+00,  8.8054e-01,  1.5542e+00,
          6.2662e-01, -1.7549e-01,  9.8284e-02, -9.3507e-02,  2.6621e-01,
         -5.8504e-01],
        [ 2.5529e-01

In [5]:
for i in embedded_sentence:
    print(i.shape)

torch.Size([16])
torch.Size([16])
torch.Size([16])
torch.Size([16])
torch.Size([16])
torch.Size([16])
torch.Size([16])
torch.Size([16])


In [6]:
omega = torch.empty(8,8)

for i, x_i in enumerate(embedded_sentence):
    for j, x_j in enumerate(embedded_sentence):
        omega[i,j] = torch.dot(x_i,x_j)

while the above code is easy to read and understand, for loops can be very inefficient, so we compute this using matrix multiplication instead....

In [8]:
embedded_sentence.shape

torch.Size([8, 16])

In [9]:
omega_mat = embedded_sentence.matmul(embedded_sentence.T)

we can use the torch.allclose function to check that this matrix above produces the expected results. if 2 tensors contain the same values, torch.allclose returns true..

In [10]:
torch.allclose(omega_mat, omega)

True

we have learned how to compute the similarity-based weights for the ith input and all inputs in the sequence (x1 to xT), the raw weights (wi1 to wiT). we can obtain the attention weights, alpha ij by normalizing the abive values via the familiar softmax function as follows below: 

In [11]:
import torch.nn.functional as F

In [12]:
omega.shape

torch.Size([8, 8])

In [13]:
attention_weights = F.softmax(omega, dim=1)
attention_weights.shape

torch.Size([8, 8])

Note that attention_weights is an 8x8 matrix, where each element represents an attention weight alphaij. for instance, if we are procesing the ith input word, the ith row of this matrix contains the corresponding attention weights for all words in the sentence. these attention weights indicate how relevant each word is to the ith word.. Hence, the columns in the attention matrix should sum up to 1, which we can confirm via the following code:

In [14]:
attention_weights.sum(dim=1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

Seen how to compute the attention weights, lets now recap and summarise:
- for a given input element, x(i), and each jth element in the set {1,...,T}, compute the dot product, x(i)T x(j)
- obtain the attention weight, alphaij, by normalizing the dot products using the softmax function
- compute the output, z(i), as the weighted sum over the entire input sequence:

Code example for computing context vectors, as the attention-weighted sum of inputs -- in particular, lets assume we are computing the context vector for the second input word

In [15]:
x_2 = embedded_sentence[1,:]
context_vec_2 = torch.zeros(x_2.shape)

In [17]:
for j in range(8):
    x_j = embedded_sentence[j, :]
    context_vec_2 += attention_weights[1,j] * x_j

context_vec_2

tensor([-9.3975e-01, -4.6856e-01,  1.0311e+00, -2.8192e-01,  4.9373e-01,
        -1.2896e-02, -2.7327e-01, -7.6358e-01,  1.3958e+00, -9.9543e-01,
        -7.1287e-04,  1.2449e+00, -7.8077e-02,  1.2765e+00, -1.4589e+00,
        -2.1601e+00])

again, we can achieve the above more efficiently using matrix multiplication. Using the following code, we are computing the context vectors for all 8 input words..

In [18]:
context_vectors = torch.matmul(
    attention_weights,
    embedded_sentence
)

Similar to the input word embeddings in embedded_sentence, the context vectors matrix has a dimensionality of 8x16. the second row in this matrix contains the context vector for the second input word.

**Parameterizing the self-attention mechanism: scaled dot product attention:**
- this next sub summarises the more advanced self-attention mechanism known as scaled dot-product attention used in the transformer architecture.
- to make the self attention mechanism more flexible and amenable to model optimization, we introduce 3 additional weight matrices that can be fit as model params during the training process: Uq, Uk, and Uv. They are used to project the inputs into query, key, and value sequence elements, as follows:

In [19]:
torch.manual_seed(123)

In [20]:
d = embedded_sentence.shape[1]

In [21]:
d

16

In [22]:
# introducing the parameterised matrices
U_query = torch.rand(d,d)
U_key = torch.rand(d,d)
U_value = torch.rand(d,d)

Using the query projection matrix, we can then compute the query sequence. for this example, consider the second input element x(i) as the query

In [23]:
x_2 = embedded_sentence[1]

In [25]:
query_2 = U_query.matmul(x_2)

In [26]:
query_2.shape

torch.Size([16])

in similar fashion, we can compute the key and value sequences, k(i) and v(i)

In [27]:
key_2 = U_key.matmul(x_2)
value_2 = U_value.matmul(x_2)

However, as seen from prev examples, we also need the key and value sequences for all other input elements, which we can compute as follows

In [28]:
keys = U_key.matmul(embedded_sentence.T).T
values = U_value.matmul(embedded_sentence.T).T

In [29]:
keys.shape, values.shape

(torch.Size([8, 16]), torch.Size([8, 16]))

- from the prev section, we computed the unnormalized weights as the pairwise dot product between the given input sequence element, x(i), and the jth sequence element, x(j). Now, in this parameterized version of self attention, we compute this as the dot prod between the query and key values:
- for example, the following code computes the unnormalized attention weight, w23, i.e. the dot product between our query and the third input sequence element

In [30]:
omega_23 = query_2.dot(keys[2])
omega_23

tensor(14.3667)

since we will be needing these later, we can scale up this computation to all keys...

In [31]:
query_2.shape, keys.shape

(torch.Size([16]), torch.Size([8, 16]))

In [32]:
omega_2 = query_2.matmul(keys.T)
omega_2

tensor([-25.1623,   9.3602,  14.3667,  32.1481,  53.8976,  46.6626,  -1.2131,
        -32.9391])

next step in self-attention is to go from the unnormalized attention weights, to the normalized attention weights using the softmax function. we can then further use 1/sqrt(m) to scale the weights before normalizing it via the softmax function as follows:
- alphaij = softmax(weights/sqrt(m))
- by scaling by 1/sqrt(m), where typically m=dk, ensures the euclidean length of the weight vectors will be approx the same range.

In [33]:
attention_weights_2 = F.softmax(omega_2/d**0.5,dim=0)
attention_weights_2

tensor([2.2317e-09, 1.2499e-05, 4.3696e-05, 3.7242e-03, 8.5596e-01, 1.4026e-01,
        8.8897e-07, 3.1936e-10])

**Attention is all we need: the original architecture:**

**Encoding context embeddings via multi-head attention:**
- overall goal of the encoder block is to take in a sequential input X, and map it into a continuous representation Z, that is then passed on to the decoder
- Encoder is a stack of 6 identical layers. (6 merely being a hyperparam from the original paper). Inside each of these identical layers, there are 2 sublayers: once computes the multi-head self attention, the other is a fully connected layer.
- First - multihead self attention. In self attention used 3 matrices corresponding to the query, key, and value to transform the input sequence. In the context of multihead attention, we can think of this single operation as one attention head. 

- To illustrate the multi-head self attention stack in code consider the following:

In [34]:
torch.manual_seed(123)

In [35]:
d = embedded_sentence.shape[1]

In [36]:
one_U_query = torch.rand(d,d)

now assume we have 8 attention heads similar to the original transformer,

In [37]:
h = 8
multihead_U_query = torch.rand(h, d, d)
multihead_U_key = torch.rand(h, d, d)
multihead_U_value = torch.rand(h, d, d)

after initializing the projection matrices, we can compute the projected sequences similar to how its done in scaled dot-product attention. 

In [38]:
x_2.shape

torch.Size([16])